# Forecasting 3 — Final Assignment

Business context: Sports retailer
Problem: Predict weekly revenue for each store to help with staff and stock planning.
Dataset: `group_03_forecasting_3.csv`

We follow the six CRISP-DM phases. The numbers and graphs below come from our data and Python code.


## Phase 1 — Business Understanding

Store managers need a sales forecast before they order stock and plan staff. We assume that Sunday evening would be too late, because the case does not say when orders and staff schedules are decided. For this assignment, we choose Thursday before the target week as the planning day. This is our assumption, not a confirmed company deadline.

For example, on Thursday in week 10, we predict sales for week 11. Week 10 is not finished yet. The latest complete sales week is week 9. This means we must use sales from at least two weeks before the target week.

We predict weekly revenue (`sales_eur`) for each store. One row should represent one store-week, identified by `store_id` and `week_start`. We do not have product quantities in the dataset. Revenue forecasts can support general planning, but they cannot tell managers exactly how many products to order or staff members to schedule.

Our original success goal is at least 5% lower MAE than the two-week naive baseline (`lag_2`), without worse MAE for any store. We chose this as our own practical improvement target, not as a company rule. We keep this original goal even if later results show that another baseline is stronger. We do not have actual ordering deadlines or business costs. For this assignment, we assume Thursday is early enough for planning.

### What the error measures mean for the business

MAE tells us how many euros our forecast is wrong by on average for one store in one week. For example, an MAE of €2,000 means the average size of the error is €2,000. Some forecasts can be much closer and others much further away. It does not mean the store loses €2,000.

MAE treats a forecast that is €2,000 too high the same as one that is €2,000 too low. Bias helps us see whether forecasts are too high or too low on average. RMSE gives more weight to large mistakes. WAPE compares the total absolute error with the total actual revenue and shows this as a percentage. We also check each store because an overall result can hide large mistakes in one store.

A forecast that is too low could lead to too little stock or too few staff. A forecast that is too high could lead to unnecessary stock or staffing costs. We cannot calculate these costs or convert the errors into staff hours from this dataset. We do not have the product quantities, staffing information or cost data needed for that. Our 5% improvement target is our own choice for this assignment; it is not a proven profit target.


In [ ]:
import sys
import hashlib
from importlib.metadata import version
import sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

print('Python:', sys.version)
for package in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'scipy', 'ipykernel']:
    print(f'{package}=={version(package)}')
DATA = Path('group_03_forecasting_3.csv')
if not DATA.exists():
    raise FileNotFoundError('Place group_03_forecasting_3.csv beside this notebook and start Jupyter in that folder.')
print('CSV SHA-256:', hashlib.sha256(DATA.read_bytes()).hexdigest())
raw = pd.read_csv(DATA)
print('Dataset shape:', raw.shape)
display(raw.head())


## Phase 2 — Data Understanding

We first look at the data to understand what we are trying to predict. Each row should show the sales of one store in one week. Our target is `sales_eur`.

We check how many stores and weeks are in the data. We also look for missing values, repeated store-weeks and differences in weekly sales. This helps us decide what we need to prepare before making a forecast.

We also check which columns we can use on Thursday, before the week we want to predict. We should not use information that is only known after that week has finished.

### Variable meanings and availability

| Variable | Meaning and use |
|---|---|
| `store_id` | Store identifier; known before forecasting and used as one yes/no column per store. |
| `week_start` | Start of the revenue week; known calendar information and part of the key. |
| `sales_eur` | Recorded weekly revenue in euros; this is our target. Only completed earlier weeks can be used as inputs or as training examples. |
| `promo_label` | Type of promotion recorded for that week. We assume this column does not show a promotion plan that is available on our Thursday planning day, so we do not use it to predict sales. |
| `holiday_week` | Holiday indicator; known in advance, but we did not test it as an extra input in this iteration. It is a candidate for the next iteration. |
| `temperature_actual`, `rain_mm_actual` | Temperature and rainfall measured during the week we want to predict. That week has not happened yet on our planning day, so we cannot use these measured values. A weather forecast available on Thursday could be useful, but it is not included in this dataset. |
| `footfall_actual` | Number of visitors in that week; not used as an input. We only use it to check the quality of the sales data. |
| `average_price_eur` | Average price recorded for the week. We assume this is only available after the week, so we do not use it as an input. |
| `returns_next_week_eur` | Returns in the following week; future information, not used. |
| `manager_sales_band_after_week` | Sales band given after the week; information from after the outcome, not used. |

We use the column names to understand what the values mean. For this assignment, we assume the actual weekly values are recorded after that week and that a completed week's revenue is available before the following Thursday. These are our assumptions, not extra facts supplied with the dataset.

### Why we leave out actual weather

Weather could affect shopping. For example, rain might make an indoor shopping centre more attractive, while good weather might bring more people to an outdoor shopping area. These are possible explanations, not patterns we have tested here. We also do not have store-location information showing which stores are indoors or in city centres.

We leave these weather columns out because they contain measured weather from the future week, not because weather is irrelevant. Using the actual rain or temperature would give the model information we would not have on Thursday. In another version, we could test weather forecasts that were already available on the planning day. We cannot treat the actual weather columns as if they were those forecasts.


In [ ]:
df = raw.copy()
df['week_start'] = pd.to_datetime(df['week_start'], dayfirst=True, errors='coerce')
print('Date range:', df.week_start.min(), 'to', df.week_start.max())
print('Stores:', df.store_id.nunique(), '| weeks:', df.week_start.nunique())
print('Missing by field:'); display(df.isna().sum().to_frame('missing'))
print('Duplicate store-week rows:', df.duplicated(['store_id','week_start']).sum())
print('Promo raw labels:'); display(df.promo_label.value_counts(dropna=False).to_frame('count'))
print('Sales distribution:'); display(df.sales_eur.describe().to_frame('sales_eur'))
print('Non-positive sales:', (df.sales_eur <= 0).sum())
print('Date parse failures:', df.week_start.isna().sum())
print('Five highest sales rows:')
display(df.nlargest(5,'sales_eur')[['store_id','week_start','sales_eur']])


In [ ]:
# Avoid counting repeated store-weeks twice in descriptive totals.
descriptive_panel = df.groupby(['store_id', 'week_start'], as_index=False).agg(sales_eur=('sales_eur', 'mean'))
fig, axes = plt.subplots(1,2,figsize=(13,4))
df.sales_eur.hist(ax=axes[0],bins=35)
axes[0].set(title='Store-week sales distribution',xlabel='Sales (EUR)',ylabel='Rows')
descriptive_panel.groupby('week_start').sales_eur.sum().sort_index().plot(ax=axes[1])
axes[1].set(title='Total weekly sales (descriptive only)',xlabel='Week',ylabel='Sales (EUR)')
plt.tight_layout(); plt.show()
display(df.groupby('store_id').sales_eur.agg(['count','mean','std','min','max']).round(2))


### Phase 2 findings and how they affect our business problem

Dataset facts: The dataset has 1,065 rows, 9 stores and 118 unique weeks. Three store-week keys appear twice. `promo_label` has 71 missing values and `average_price_eur` has 18 missing values. Store revenue levels are different, and the table above shows unusual high and low weeks.

How this affects our business problem: Phase 1 asks for one weekly revenue forecast per store. The duplicate keys challenge that requirement, so we must create one row per store-week. Different store sales levels mean we must also check errors for each store. Because we forecast on Thursday before the target week, actual footfall, weather and target-week sales cannot be used as inputs.

Our assumptions and choices: Thursday is our proposed planning moment, not a confirmed company deadline. We choose to keep unusual sales values unless they are verified as wrong. We use revenue because the dataset does not provide product quantities; the forecast cannot directly tell managers how many items to order.

Unusual values: Store S09 has a recorded weekly revenue of €3,791.04 in one week, and the raw dataset contains 4 store-weeks with revenue above €60,000. The sales-per-visitor check below shows that normal revenue per visitor is about €5 in every store, while the four highest weeks have about €19 to €20 per visitor and the low S09 week has €0.50. This is a reason to investigate these weeks, but it does not prove that the values are wrong, and we do not remove them automatically.


In [ ]:
# Inspect unusually high and low revenue values. This is a data-quality check only.
inspect_cols = ['store_id','week_start','sales_eur','footfall_actual','promo_label','holiday_week']
unusual = pd.concat([df.nlargest(4,'sales_eur'),df.nsmallest(1,'sales_eur')])[inspect_cols].copy()
unusual['sales_per_visitor'] = (unusual.sales_eur / unusual.footfall_actual).round(2)
print('Unusual high and low store-weeks:')
display(unusual)
print('Median sales per visitor by store:')
display((df.sales_eur / df.footfall_actual).groupby(df.store_id).median().round(2))


## Phase 3 — Data Preparation

We make three substantive preparation decisions. For each decision, we show the evidence, technique, before/after check, interpretation and next decision. The code directly below prints the checks.

Decision 1 — Duplicate store-weeks
- Evidence: Phase 2 finds 1,065 source rows and 3 extra duplicate store-week rows (6 records involved).
- Technique: As discussed with the teacher, group by `store_id` and `week_start` and take the mean of `sales_eur` only. `returns_next_week_eur` is not averaged or used.
- Before/after: The code shows 1,065 rows before and 1,062 after, with zero repeated keys. In each duplicate pair, `sales_eur` is the same, so its mean does not change the recorded revenue.
- Interpretation: We now have one target value per store-week. This does not tell us which of the two different returns values is correct.
- Next decision: Use the resulting store-week table for the forecast and keep the different returns values marked as an open data-quality issue.

Decision 2 — Missing and unusual values
- Evidence: There are 71 missing promotion labels, 18 missing average prices, and some unusually high and low sales values.
- Technique: Standardise promotion spelling and fill missing labels with `unknown`. Exclude promotions and average price from the inputs because we assume these recorded weekly values are not available on our Thursday planning day. Keep unusual revenue values in the main analysis; inspect them instead of silently deleting them.
- Before/after: The code prints promotion categories before and after standardisation, plus the original missing-price count. The number of missing promotion labels falls from 71 to zero because we use the category `unknown`; this does not recover the missing promotion information. The revenue target is not changed by this decision.
- Interpretation: This avoids using target-week information that would not be available. Unusual values may be real or errors; the dataset alone cannot prove which.
- Next decision: Keep unusual sales marked for further investigation; we do not assume they are wrong. A sensitivity check later in the notebook tests, on validation data only, whether unusual training values affect the model. It is not used to pick a better-looking model from the test results.

Decision 3 — Available historical features
- Evidence: On Thursday, the current week is unfinished, so the sales of the week just before the target week (`lag_1`) are not yet known. One row per store-week is required for time shifts.
- Technique: Sort by store and week, create `lag_2`, `lag_3`, `lag_5`, `lag_9`, a four-week rolling mean of completed weeks, and calendar features. Exclude actual target-week footfall, weather, returns and other future information. `holiday_week` is known in advance, but we did not test it as an extra input in this iteration.
- Before/after: The code prints sample shifted rows and the number of complete feature rows: 1,062 prepared store-weeks become 981 modelling rows because the first weeks do not have enough history.
- Interpretation: All selected sales inputs come from fully completed weeks before the Thursday prediction.
- Next decision: Use time-ordered training, validation and test periods. Any feature changes must be decided using training and validation data, never by looking at which choice wins on test.

### Why we made these choices

We do not know why some promotion labels are missing. A missing label does not prove that there was no promotion, so we use `unknown` and not `none`. We do not fill in the missing average prices, because price is not used in the model and we do not know why these values are missing.

In the three duplicate pairs, the revenue values are the same. The average is therefore the same as either recorded value, while adding them together would count the same week twice. This solution fits this data; it does not mean that averaging is always the right way to handle conflicting sales records.

All lag and rolling calculations are made separately for each store after sorting by date, so a store never uses another store's sales history. `lag_3` adds the week before the most recent available week, and `lag_5` and `lag_9` add information from roughly one and two months earlier. The four-week average uses weeks t−2 to t−5. It smooths out week-to-week changes, but it reacts slowly to real changes and can still be affected by unusual weeks. These are reasonable starting choices; we have not proved that they are the best ones.

The model uses the sales history features, the month, two features that describe the position of the week in the year (`week_sin` and `week_cos`) and one yes/no column for each store. We do not use the promotion label or a season column. The two week features make sure that the last week of December and the first week of January are treated as close to each other.


In [ ]:
before = len(df)
conflicts = df.loc[df.duplicated(['store_id','week_start'],keep=False)].sort_values(['store_id','week_start'])
print('Duplicate-key source rows:'); display(conflicts)
print('Missing promotion labels before:', df.promo_label.isna().sum())
print('Raw promotion categories:', df.promo_label.nunique(dropna=True))
df['promo_clean'] = df.promo_label.astype('string').str.strip().str.lower().fillna('unknown')
print('Promo categories after standardization:'); display(df.promo_clean.value_counts())
print('Missing cleaned promotion labels after:', df.promo_clean.isna().sum())
print('Missing prices before exclusion:', df.average_price_eur.isna().sum())
# Provisional deduplication; non-target fields are not used as inputs.
panel = (df.groupby(['store_id','week_start'],as_index=False)
         .agg(sales_eur=('sales_eur','mean'))
         .sort_values(['store_id','week_start']).reset_index(drop=True))
assert not panel.duplicated(['store_id','week_start']).any()
print('Rows before / after:', before, len(panel))
print('Key duplicates after:',panel.duplicated(['store_id','week_start']).sum())
print('Dates per store:',panel.groupby('store_id').week_start.nunique().to_dict())
week_steps = panel.groupby('store_id').week_start.diff().dropna()
assert week_steps.eq(pd.Timedelta(days=7)).all(), 'Reindex missing weeks before creating row-based lags.'
print('Verified: consecutive observations are exactly seven days apart for every store.')


In [ ]:
panel['week_number'] = panel.week_start.dt.isocalendar().week.astype(int)
panel['month'] = panel.week_start.dt.month
panel['week_sin'] = np.sin(2*np.pi*panel.week_number/52.18)
panel['week_cos'] = np.cos(2*np.pi*panel.week_number/52.18)
g = panel.groupby('store_id',sort=False)['sales_eur']
for lag in [2,3,5,9]:
    panel[f'lag_{lag}'] = g.shift(lag)
panel['trailing_4_mean'] = panel.groupby('store_id',sort=False)['sales_eur'].transform(lambda x: x.shift(2).rolling(4).mean())
print('Before/after check: the first forecast uses week t-2 or older:')
display(panel.loc[panel.store_id == panel.store_id.iloc[0],['week_start','sales_eur','lag_2','lag_3','trailing_4_mean']].head(12))
model_data = panel.dropna(subset=['lag_2','lag_3','lag_5','lag_9','trailing_4_mean']).copy()
print('Rows available after creating older sales features:',len(model_data))
assert model_data[['lag_2','lag_3','lag_5','lag_9','trailing_4_mean']].notna().all().all()


## Phase 4 — Modelling

We predict revenue on Thursday before the target week. We cannot use sales from the current unfinished week. We compare two simple methods with a machine learning model:

- Two-week naive baseline (`lag_2`): use the revenue from two weeks before the target week. This is the baseline used in our original 5% success criterion.
- Four-week average (`trailing_4_mean`): use the average revenue from the four most recent fully completed weeks. This smooths out week-to-week changes.
- HistGradientBoosting model: use older sales, the store and the time of year. `lag_5` and `lag_9` add information from roughly one and two months before the target week, but we have not proved that these particular lags are the best ones. We test 4, 8 and 16 maximum leaf nodes. Leaf nodes are the endpoints of the small decision trees; more leaf nodes allow more detailed patterns but do not guarantee better results.

We train on older dates, choose the model settings using later validation dates, and evaluate on the final test dates. The test is a rolling Thursday forecast: earlier completed weeks can be used when they become available. It is not one fixed forecast for all test weeks.

Important: choices about features, unusual values and model settings must be based on training and validation data, not on the final test results. We use the test period for the final evaluation and for diagnostic checks, not to select a better-looking model afterwards.

We use HistGradientBoosting because it can find patterns that combine sales history, store and time of year. We keep the model small by testing only 4, 8 and 16 leaf nodes. We only have about two years of data, so the model could learn patterns that do not repeat. That is why the simple baselines are important, and the model does not have to be the winner.

Training data and forecast timing: before the first Thursday of the validation period and of the test period, the model may only learn from weeks that are already complete on that Thursday. We assume a week that starts on Monday is complete and available seven days later. This means the week just before the target week cannot be used for training. The model stays the same during each period, but its sales history inputs update every week as new complete weeks become available. Store columns are created from the stores in the training data.

We had already looked at the test results when we found the training timing mistake. We fixed this mistake and ran the evaluation again. We did not use the test results to choose different features or model settings.

In [ ]:
dates = np.array(sorted(model_data.week_start.unique()))
cut1, cut2 = dates[int(len(dates)*0.65)], dates[int(len(dates)*0.80)]
validation_origin = pd.Timestamp(cut1) - pd.Timedelta(days=4)
test_origin = pd.Timestamp(cut2) - pd.Timedelta(days=4)
train = model_data[model_data.week_start + pd.Timedelta(days=7) <= validation_origin].copy()
val = model_data[(model_data.week_start >= cut1)&(model_data.week_start < cut2)].copy()
test = model_data[model_data.week_start >= cut2].copy()
print('First forecast origins:', validation_origin.date(), test_origin.date())
print('Cutoffs:',pd.Timestamp(cut1).date(),pd.Timestamp(cut2).date())
for label,part in [('train',train),('validation',val),('test',test)]:
    print(label,len(part),part.week_start.min().date(),part.week_start.max().date())
assert train.week_start.max() < val.week_start.min() < test.week_start.min()
base_features = ['lag_2','lag_3','lag_5','lag_9','trailing_4_mean','month','week_sin','week_cos']
stores = sorted(train.store_id.unique())
assert set(val.store_id).issubset(stores) and set(test.store_id).issubset(stores)
assert (train.week_start + pd.Timedelta(days=7) <= validation_origin).all()
store_columns = ['store_'+s for s in stores]
def features(frame):
    x = frame[base_features].copy()
    for s,c in zip(stores,store_columns):
        x[c] = (frame.store_id == s).astype(int)
    return x

def metrics(y,p):
    y=np.asarray(y);p=np.asarray(p)
    return {'MAE_EUR':mean_absolute_error(y,p),
            'RMSE_EUR':np.sqrt(mean_squared_error(y,p)),
            'WAPE_pct':100*np.abs(y-p).sum()/np.abs(y).sum(),
            'Bias_EUR':np.mean(p-y)}

results=[]
for depth in [2,3,4]:
    candidate=HistGradientBoostingRegressor(max_iter=150,max_leaf_nodes=2**depth,
                    learning_rate=0.05,l2_regularization=10,random_state=42)
    candidate.fit(features(train),train.sales_eur)
    pred=candidate.predict(features(val))
    results.append({'max_leaf_nodes':2**depth,**metrics(val.sales_eur,pred)})
validation_results=pd.DataFrame(results).sort_values('MAE_EUR')
print('Candidate validation results:');display(validation_results.round(2))
print('Simple baselines on validation:')
display(pd.DataFrame({'Two-week naive':metrics(val.sales_eur,val.lag_2), 'Four-week average':metrics(val.sales_eur,val.trailing_4_mean)}).T.round(2))
best_leaves=int(validation_results.iloc[0].max_leaf_nodes)
print('Chosen leaves using validation only:',best_leaves)


### Validation-only sensitivity check (not a final model choice)

We check whether unusually high or low training revenue values affect the model. We calculate each store's typical revenue per visitor from training data only. For this check, we flag training rows with less than half or more than twice that typical value. These limits are only for investigation; they do not prove that a value is wrong. We train a second model without those training rows and compare both versions on validation data only. We keep the original model settings and do not choose a version because of its test score. Removing a training row does not remove its value from later historical inputs, which limits this check.


In [ ]:
# Validation-only sensitivity analysis: thresholds estimated from training weeks only.
# Use the original source for footfall; duplicated source keys have identical footfall.
footfall_lookup = df.groupby(['store_id','week_start']).footfall_actual.first()
train_footfall = np.array([footfall_lookup.loc[(s,w)] for s,w in zip(train.store_id,train.week_start)])
ratio = train.sales_eur.to_numpy() / train_footfall
reference = pd.Series(ratio,index=train.index).groupby(train.store_id).median()
relative = ratio / train.store_id.map(reference).to_numpy()
flagged_train = (relative > 2) | (relative < 0.5)
print('Flagged training rows (for sensitivity only):',int(flagged_train.sum()))
display(train.loc[flagged_train,['store_id','week_start','sales_eur']])
original_validation_model = HistGradientBoostingRegressor(max_iter=150,max_leaf_nodes=best_leaves,learning_rate=0.05,l2_regularization=10,random_state=42)
original_validation_model.fit(features(train),train.sales_eur)
sensitivity_validation_model = HistGradientBoostingRegressor(max_iter=150,max_leaf_nodes=best_leaves,learning_rate=0.05,l2_regularization=10,random_state=42)
sensitivity_validation_model.fit(features(train.loc[~flagged_train]),train.loc[~flagged_train,'sales_eur'])
sensitivity_results = pd.DataFrame({
 'Original candidate':metrics(val.sales_eur,original_validation_model.predict(features(val))),
 'Candidate excluding flagged training targets':metrics(val.sales_eur,sensitivity_validation_model.predict(features(val))),
 'Four-week average':metrics(val.sales_eur,val.trailing_4_mean)}).T
print('Sensitivity comparison on validation only (test not used):')
display(sensitivity_results.round(2))
print('This is diagnostic. The final candidate is still selected using the original validation procedure.')
print('Validation MAE difference, original minus sensitivity (EUR):', round(sensitivity_results.loc['Original candidate','MAE_EUR'] - sensitivity_results.loc['Candidate excluding flagged training targets','MAE_EUR'],2))


In [ ]:
original = sensitivity_results.loc['Original candidate']
sensitive = sensitivity_results.loc['Candidate excluding flagged training targets']
four = sensitivity_results.loc['Four-week average']
change_pct = 100 * (sensitive.MAE_EUR / original.MAE_EUR - 1)
vs_four_pct = 100 * (sensitive.MAE_EUR / four.MAE_EUR - 1)
display(Markdown(f"""### Interpretation of the sensitivity check

When we leave out the {int(flagged_train.sum())} flagged training rows, the validation MAE changes from €{original.MAE_EUR:,.2f} to €{sensitive.MAE_EUR:,.2f}, which is {abs(change_pct):.1f}% {'lower' if change_pct < 0 else 'higher'}. The four-week average has a validation MAE of €{four.MAE_EUR:,.2f}, so the model without the flagged rows is {abs(vs_four_pct):.1f}% {'worse' if vs_four_pct > 0 else 'better'} than the four-week average.

The bias changes from {original.Bias_EUR:+,.2f} euros to {sensitive.Bias_EUR:+,.2f} euros. A negative bias means that the forecast is too low on average. A lower MAE therefore does not mean that every part of the forecast improved.

This does not prove that the flagged records are wrong, and it does not justify deleting them. Their values also still appear in later historical inputs, so this is not a complete cleaning test. We keep the original model and leave any cleaning rule for a future training and validation cycle. The test results are not used to make this decision.
"""))


## Phase 5 — Evaluation

We compare the model with both simple baselines on the held-out test weeks. We check MAE, RMSE, WAPE, bias and the results for each store. Lower MAE is better. Here bias = prediction minus actual revenue, so a negative bias means that revenue was predicted too low.

A forecast that is too low could lead to too little stock or too few staff, while a forecast that is too high could lead to unnecessary stock or staffing costs. Revenue does not directly measure the number of products or workers needed, and we have no cost data. These are possible business effects, not measured losses.

We keep unusual sales values in the main analysis because we do not have enough evidence to say they are wrong. We investigate them, but do not automatically treat them as errors.

We use the test results to evaluate the model and understand its mistakes. We do not use these results to choose new features, remove records or tune the model. Any changes to the approach would first need to be checked on validation data.

In [ ]:
trainval = model_data[model_data.week_start + pd.Timedelta(days=7) <= test_origin].copy()
assert (trainval.week_start + pd.Timedelta(days=7) <= test_origin).all()
print('Final training labels:', len(trainval), trainval.week_start.min().date(), 'to', trainval.week_start.max().date())
final_model=HistGradientBoostingRegressor(max_iter=150,max_leaf_nodes=best_leaves,
                    learning_rate=0.05,l2_regularization=10,random_state=42)
final_model.fit(features(trainval),trainval.sales_eur)
eval_df=test[['store_id','week_start','sales_eur']].copy()
eval_df['naive']=test.lag_2.to_numpy()
eval_df['four_week_average']=test.trailing_4_mean.to_numpy()
eval_df['candidate']=final_model.predict(features(test))
comparison=pd.DataFrame({'Naive two weeks ago':metrics(eval_df.sales_eur,eval_df.naive),
                         'Four-week average':metrics(eval_df.sales_eur,eval_df.four_week_average),
                         'Candidate model':metrics(eval_df.sales_eur,eval_df.candidate)}).T
comparison['MAE_vs_naive_pct']=100*(comparison.MAE_EUR/comparison.loc['Naive two weeks ago','MAE_EUR']-1)
print('Final held-out test results:');display(comparison.round(2))
rows=[]
for store,grp in eval_df.groupby('store_id'):
    a=metrics(grp.sales_eur,grp.naive);b=metrics(grp.sales_eur,grp.candidate);c=metrics(grp.sales_eur,grp.four_week_average)
    rows.append({'store_id':store,'naive_MAE':a['MAE_EUR'],'candidate_MAE':b['MAE_EUR'],'four_week_MAE':c['MAE_EUR'],
                 'candidate_WAPE_pct':b['WAPE_pct'],'candidate_bias_EUR':b['Bias_EUR'],
                 'MAE_change_pct':100*(b['MAE_EUR']/a['MAE_EUR']-1),
                 'candidate_vs_four_week_pct':100*(b['MAE_EUR']/c['MAE_EUR']-1)})
store_results=pd.DataFrame(rows)
print('Store-level held-out results:');display(store_results.round(2))
fig,ax=plt.subplots(figsize=(13,4))
weekly=eval_df.groupby('week_start')[['sales_eur','naive','four_week_average','candidate']].sum()
weekly.plot(ax=ax);ax.set(title='Held-out period: actual vs aggregate store forecasts',ylabel='EUR',xlabel='Week');plt.tight_layout();plt.show()


In [ ]:
naive_mae=comparison.loc['Naive two weeks ago','MAE_EUR']
rolling_mae=comparison.loc['Four-week average','MAE_EUR']
model_mae=comparison.loc['Candidate model','MAE_EUR']
improvement=100*(naive_mae-model_mae)/naive_mae
model_vs_rolling=100*(model_mae/rolling_mae-1)
store_worse=(store_results.MAE_change_pct>0).sum()
print(f'Test MAE improvement versus two-week naive: {improvement:.1f}%.')
print(f'Candidate MAE relative to four-week average: {model_vs_rolling:+.1f}% (positive means worse).')
print(f'Stores with worse candidate MAE than two-week naive: {store_worse} of {len(store_results)}.')
print('Original success criterion: at least 5% lower MAE than two-week naive and no store worse.')
print('Original criterion met:',improvement >= 5 and store_worse == 0)
if model_mae >= rolling_mae:
    print('Business decision: do not deploy the current candidate. The four-week average is stronger on this held-out period. Review data quality and return to preparation/validation.')
elif improvement >= 5 and store_worse == 0:
    print('Business decision: a controlled shadow pilot may be considered, with manager review.')
else:
    print('Business decision: do not deploy the current candidate. Review store-level errors and validation evidence first.')
print('Test-period comparisons are final evaluation and diagnostics, not a reason to tune the model on test data.')

print('Stores with worse candidate MAE than four-week average:', int((store_results.candidate_MAE > store_results.four_week_MAE).sum()), 'of',len(store_results))
print('Candidate overall bias (EUR):',round(comparison.loc['Candidate model','Bias_EUR'],2))


In [ ]:
worse_naive = store_results.loc[store_results.candidate_MAE > store_results.naive_MAE, 'store_id'].tolist()
worse_four = store_results.loc[store_results.candidate_MAE > store_results.four_week_MAE, 'store_id'].tolist()
validation_naive = metrics(val.sales_eur, val.lag_2)['MAE_EUR']
validation_four = metrics(val.sales_eur, val.trailing_4_mean)['MAE_EUR']
validation_model = validation_results.iloc[0].MAE_EUR
model_bias = comparison.loc['Candidate model','Bias_EUR']
criterion_met = improvement >= 5 and store_worse == 0
n_stores = len(store_results)

if validation_model > max(validation_naive, validation_four):
    validation_sentence = 'Both simple methods were therefore already more accurate than the model before the final test.'
else:
    validation_sentence = 'The model was not clearly behind both simple methods before the final test.'

if model_mae >= rolling_mae:
    decision_sentence = ('we do not deploy the current model. The four-week average is the stronger forecast. '
                         'We return to Data Preparation, check the suspicious records, and confirm the planning deadline and when weekly revenue becomes available. '
                         'A revised model must first beat the four-week average on new validation data and in a new test period before it is used.')
elif criterion_met:
    decision_sentence = 'a controlled shadow pilot with manager review may be considered, as described in Phase 6.'
else:
    decision_sentence = ('we do not deploy the current model. We first review the store-level errors and the validation results, '
                         'and test a revised model on new validation data.')

display(Markdown(f"""### Interpretation and remaining limits

On the test weeks, the model has an MAE of €{model_mae:,.2f}, compared with €{naive_mae:,.2f} for the two-week baseline and €{rolling_mae:,.2f} for the four-week average. The model is {improvement:.1f}% better than the two-week baseline, but {abs(model_vs_rolling):.1f}% {'worse' if model_vs_rolling > 0 else 'better'} than the four-week average. Our original success criterion is {'met' if criterion_met else 'not met'}: we wanted at least 5% lower MAE than the two-week baseline and no store with a worse MAE. We keep this original criterion.

The model is worse than the two-week baseline in {len(worse_naive)} of {n_stores} stores ({', '.join(worse_naive) or 'none'}) and worse than the four-week average in {len(worse_four)} of {n_stores} stores ({', '.join(worse_four) or 'none'}). The store table above shows the exact MAE for each store. These store results are a diagnostic check; we do not use them to choose a different method for each store.

On validation, before the final test, the MAE was €{validation_naive:,.2f} for the two-week baseline, €{validation_four:,.2f} for the four-week average and €{validation_model:,.2f} for the best model setting. {validation_sentence}

The overall bias of the model is {model_bias:+,.2f} euros (prediction minus actual), so on average the forecast is {'too low' if model_bias < 0 else 'too high'}. A forecast that is too low could lead to too little stock or too few staff in busy weeks. A forecast that is too high could lead to unnecessary stock or staffing costs. These are possible effects, not measured losses. Revenue does not show exact product quantities or staffing needs, and recorded revenue can be lower than real demand if products were sold out.

Decision: {decision_sentence}

Limits: we tested only one final period, and the unusual revenue values are not verified. Promotion plans could be tested in a future cycle if they are known by Thursday.
"""))

# Diagnose the observed low week and its later use in the naive forecast.
s09 = eval_df.loc[eval_df.store_id == 'S09'].copy()
s09['naive_absolute_error'] = (s09.naive - s09.sales_eur).abs()
diagnostic = s09.loc[s09.week_start.isin(pd.to_datetime(['2026-03-02', '2026-03-16']))]
print('S09 diagnostic only: low recorded week and its lag_2 effect two weeks later')
display(diagnostic[['week_start','sales_eur','naive','four_week_average','candidate','naive_absolute_error']].round(2))
display(Markdown('The low S09 week affects the two-week baseline twice: first as the actual outcome, and two weeks later as its forecast. This explains why the comparison is sensitive to one unusual week. It does not prove that the record is wrong, and we do not remove any test records.'))


### Facts, assumptions and our choices

- Dataset facts: Nine stores, weekly `sales_eur`, missing fields, duplicate store-week records, and the validation and test results above come from the supplied data and code.
- Business assumptions: Thursday is early enough for planning; managers can use revenue forecasts to support decisions; the company can check questionable records. 
- Our analytical choices: Forecast revenue rather than units, use `lag_2` and the four-week mean as baselines, average duplicate `sales_eur`, exclude features that are not available on Thursday, use chronological splits, and keep the original 5% goal.

The final test is used to evaluate the model and make a business recommendation, not to choose new features, thresholds or model settings. Any future model revision needs a new time-based validation and a fresh final evaluation.

### Why can a simple average beat the model?

The sensitivity check shows that a few unusual training rows have a large effect on the model. The store table shows where each method works better, but it does not tell us the business reason.

Possible reasons are that sales in most stores are fairly stable, so a short average already works well; that two years of data is not much for the model to learn yearly patterns; and that tree-based models can be strongly affected by unusual values. These are possible explanations, not proven causes. We did not run the extra analyses that would be needed to prove them.

The model was tuned in a limited way: we compared 4, 8 and 16 leaf nodes on validation and used fixed values for the other settings (150 iterations, learning rate 0.05 and L2 regularization 10). In a future cycle, we could test a few alternatives over several later periods, using only earlier data each time. We would not use the current test results for this.

The four-week average is simple and easy to explain, so managers could use it as a planning reference once the planning deadline and data availability are confirmed. Extra information could be tested in the future, but only if it is known on Thursday, for example planned promotions or weather forecasts, not the actual weather or footfall of the target week.


## Phase 6 — Deployment / Next Iteration

Current decision: we do not use the machine learning model for stock or staffing decisions. It performed worse than the four-week average, so we first investigate the unusual sales values and test a revised model on new validation data. The plan below describes how a future, improved model could be used.

Who and when: an analyst would produce the forecast every Thursday for the next week, using only completed sales weeks. Store managers would use it, together with the four-week average, to plan stock and staff. Managers can change the forecast, for example for a local event, and must record the reason. The retailer must confirm that Thursday is early enough. Revenue supports general planning, but it cannot directly tell managers exact product quantities or staffing levels.

Data checks: before every forecast, we would check for missing weeks, duplicated store-weeks and unusually high or low sales. A data owner would check definitions, duplicates and unusual values. If sales history for a store is missing, we would not make the automatic forecast for that store; the manager would plan manually until the data is fixed.

Monitoring: we would save every forecast before the target week and compare it with actual revenue afterwards. We would track MAE and bias for each store and overall, and compare both with the four-week average over the same weeks. Bias is prediction minus actual. We check both underforecasting and overforecasting, because both can cause planning problems.

Shadow pilot: an improved model would first run in a shadow pilot. Its forecasts are recorded, but they do not change real orders or staffing. After eight weeks, we review the results. If the model is worse than the four-week average over these eight weeks, or if important data is unreliable, we keep using the four-week average and investigate.

Review and retraining: we would review the model and data every month, using the latest eight completed weeks. We would start a retraining investigation if the model has a higher MAE than the four-week average over these eight weeks, if a store is worse than the four-week average in two monthly reviews in a row, or if the data owner reports a major change in sales patterns or definitions. Retraining is never automatic: we first check data quality and test the updated model on later weeks before another shadow pilot. These roles, rules and the monthly schedule are our proposals, not company rules. Managers always keep the final decision.


## Individual Understanding

### Tijn

In previous assignments, I sometimes made conclusions too quickly without checking the data carefully enough. In this assignment, I learned to explain the difference between what the data shows and what we assume. For example, we chose Thursday as the planning day, but we do not know if this is the company's actual deadline. This choice means that we can only use sales information that is complete and available on that Thursday, both for the input features and for training the model.

I also learned that a better overall result does not mean the model works better for every store. Our model can beat one baseline but still perform worse than another simple method. Next time, I would check the timing of the available data earlier and look at both the overall results and the results for each store before making a recommendation.

### Jayden

In previous assignments, I learned that we need to be careful when explaining patterns in the data. This assignment showed me that the same applies to missing and unusual values. A missing promotion label does not mean there was no promotion, and a very high sales value is not automatically wrong. We should investigate these values before changing them.

I also learned to look at more than one result when comparing models. In the sensitivity check, we looked at what happened when we left unusual rows out of training. A lower MAE means smaller errors on average, but we also need to check whether predictions are usually too high or too low. Those unusual sales values can also still appear in later lag features. Next time, I would explain these limits more clearly and compare the model with simple baselines before deciding whether the business should use it.

### Daniel

In previous assignments, we sometimes changed unusual values too quickly. In our last assignment, we replaced one extreme sales value without checking whether it was really wrong, and we calculated the limit for unusual values on the full dataset before splitting the data. In this assignment, I learned to investigate a strange value before changing it. I compared the extreme weeks with footfall. Those weeks had a normal number of visitors but about four times the normal revenue per visitor, which suggests a recording error, although it does not prove one. I also learned that every rule we learn from the data should come from the training period only.

I also learned that beating one baseline is not enough. Our model was better than the two-week baseline, but the four-week average was still better overall, so the right business decision was not to deploy the model yet. Next time, I would compare the model with more than one simple method from the start and check the timing of the available data earlier.
